# Phase 3: Real-time Simulation & Deployment
## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## 2. Load Models and Scaler

In [2]:
try:
    model = joblib.load('../models/random_forest.pkl')
    scaler = joblib.load('../models/scaler.pkl')
    le = joblib.load('../models/label_encoder.pkl')
    selected_features = joblib.load('../models/selected_features.pkl')
    print("Model, Scaler, and Features loaded successfully!")
except Exception as e:
    print(f"Error loading models: {e}. Please ensure Notebook 02 has been fully run.")

Model, Scaler, and Features loaded successfully!


## 3. Real-time Prediction Function

In [3]:
def predict_flow(flow_dict):
    """
    Simulates real-time prediction for a single network flow.
    """
    try:
        # Convert input to dataframe to ensure proper column order
        df_flow = pd.DataFrame([flow_dict])
        
        # Ensure all required features exist (fill missing with 0.0)
        for f in selected_features:
            if f not in df_flow.columns:
                df_flow[f] = 0.0
                
        X_input = df_flow[selected_features]
        
        # --- PREDICTION LOGIC ---
        # If the model is a Pipeline (it contains a 'scaler' step), we don't scale manually
        # Otherwise, we apply the standalone scaler
        if hasattr(model, 'named_steps') or 'Pipeline' in str(type(model)):
            pred_encoded = model.predict(X_input)[0]
        else:
            X_scaled = scaler.transform(X_input)
            pred_encoded = model.predict(X_scaled)[0]
            
        pred_label = le.inverse_transform([pred_encoded])[0]
        
        # --- ALERTING & LOGGING ---
        if pred_label != 'BENIGN':
            # Use datetime from 'from datetime import datetime'
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            alert_msg = f"[{timestamp}] [ALERT] Suspicious traffic detected: {pred_label}\n"
            
            # Safe print for different terminal encodings
            try:
                print(f"\033[91m{alert_msg.strip()}\033[0m")
            except Exception:
                print(f"[ALERT] {pred_label}")
            
            # Enforce UTF-8 for file logging
            with open('../alerts.log', 'a', encoding='utf-8') as f:
                f.write(alert_msg)
        else:
            # Normal traffic display
            try:
                print(f"\033[92mNormal traffic (BENIGN)\033[0m")
            except Exception:
                print("Normal traffic (BENIGN)")
                
        return pred_label

    except Exception as e:
        # Handle cases where model or other artifacts aren't loaded yet
        return f"Prediction Error: {str(e)}"


## 4. Test the Simulation

In [8]:
import pandas as pd
import os
import json
from datetime import datetime

try:
    # 0. Configure paths and create result folder
    # We use absolute paths to ensure the folder is created at the project root
    project_root = os.path.dirname(os.getcwd())
    result_dir = os.path.join(project_root, 'result')
    os.makedirs(result_dir, exist_ok=True)
    
    # 1. Configure the number of test samples
    NUM_SAMPLES = 500 # 1000 tests total
    
    # 2. Load cleaned data from Notebook 01
    df_all = pd.read_parquet('../datasets/cleaned_data.parquet')
    
    # 3. Sample and Shuffle
    df_benign = df_all[df_all['Label'] == 'BENIGN'].sample(NUM_SAMPLES, random_state=42)
    df_attacks = df_all[df_all['Label'] != 'BENIGN'].sample(NUM_SAMPLES, random_state=42)
    df_sample = pd.concat([df_benign, df_attacks]).sample(frac=1, random_state=42)
    
    print(f"Starting simulation on {len(df_sample)} network flows...\n")
    print("-" * 50)
    
    # --- INITIALIZE TRACKING ---
    correct_preds = 0
    json_results = []
    
    for _, row in df_sample.iterrows():
        # Extract flow data
        flow_data = row.drop('Label').to_dict()
        true_label = row['Label']
        
        # Predict
        predicted_label = predict_flow(flow_data) 
        
        # Determine status (match or mismatch)
        status = "success" if predicted_label == true_label else "FAILURE"
        if status == "success":
            correct_preds += 1
            
        # --- GENERATE LOG MESSAGE ---
        dest_port = flow_data.get('Destination Port', 'N/A')
        
        # Only use [ALERT] if the model predicts something other than BENIGN
        if predicted_label != 'BENIGN':
            suricata_log = f"[ALERT] Suspicious traffic detected: {predicted_label}. Destination Port: {dest_port}."
        else:
            suricata_log = f"Normal traffic detected: BENIGN. Destination Port: {dest_port}."
        
        json_results.append({
            "true-label": true_label,
            "log": suricata_log,
            "status": status
        })
        
    # --- SAVE TO JSON ---
    json_path = os.path.join(result_dir, 'simulation_output.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(json_results, f, indent=4, ensure_ascii=False)
    
    # --- PRINT FINAL STATISTICS ---
    print("\n" + "-" * 50)
    print("SIMULATION COMPLETE.")
    print(f"- Total samples tested : {len(df_sample)}")
    print(f"- Total correct predictions : {correct_preds}")
    print(f"- Total false predictions : {len(df_sample) - correct_preds}")
    print(f"- Accuracy             : {(correct_preds / len(df_sample)) * 100:.2f}%")
    print(f"\nResults saved to: {json_path}")
        
except Exception as e:
    print(f"Error during simulation: {e}")


Starting simulation on 1000 network flows...

--------------------------------------------------
[2026-05-04 12:38:22] [ALERT] Suspicious traffic detected: DDoS
[2026-05-04 12:38:22] [ALERT] Suspicious traffic detected: DDoS
[2026-05-04 12:38:22] [ALERT] Suspicious traffic detected: DDoS
[2026-05-04 12:38:22] [ALERT] Suspicious traffic detected: DoS Hulk
Normal traffic (BENIGN)
[2026-05-04 12:38:22] [ALERT] Suspicious traffic detected: DDoS
[2026-05-04 12:38:22] [ALERT] Suspicious traffic detected: DDoS
[2026-05-04 12:38:22] [ALERT] Suspicious traffic detected: DoS Hulk
[2026-05-04 12:38:22] [ALERT] Suspicious traffic detected: DoS Hulk
Normal traffic (BENIGN)
[2026-05-04 12:38:22] [ALERT] Suspicious traffic detected: DDoS
Normal traffic (BENIGN)
[2026-05-04 12:38:22] [ALERT] Suspicious traffic detected: DoS Hulk
[2026-05-04 12:38:22] [ALERT] Suspicious traffic detected: DDoS
[2026-05-04 12:38:22] [ALERT] Suspicious traffic detected: PortScan
[2026-05-04 12:38:22] [ALERT] Suspicious tr